In [2]:
dataset = load_dataset("csv", data_files={"train": "data/train.csv", "test": "data/test.csv"})

In [12]:
dataset["train"]

Dataset({
    features: ['id', 'model_a', 'model_b', 'prompt', 'response_a', 'response_b', 'winner_model_a', 'winner_model_b', 'winner_tie'],
    num_rows: 57477
})

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [15]:
def preprocess(batch):
    """Preprocess function"""
    prompts = [
        f"<prompt>{prompt}\n\n<answer1>{resp_a}\n\n<answer2>{resp_b}"
        for prompt, resp_a, resp_b in zip(batch["prompt"], batch["response_a"], batch["response_b"])
    ]
    labels = [
        0 if win_a else 2 if win_b else 1
        for win_a, win_b in zip(batch["winner_model_a"], batch["winner_model_b"])
    ]
    return {**tokenizer(prompts, truncation=True), "labels": labels}

tokenized = dataset.map(preprocess, batched=True)

Map:   0%|          | 0/57477 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [5]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [6]:
id2label = {0: "Win A", 1: "Tie", 2: "Win B"}
label2id = {v:k for k, v in id2label.items()}

In [7]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels=3, id2label=id2label, label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
training_args = TrainingArguments(
    output_dir="out",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    processing_class=tokenizer,
    data_collator=data_collator
)

In [18]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.073100,1.090337
2,1.039600,1.177386


TrainOutput(global_step=7186, training_loss=1.0682368545256185, metrics={'train_runtime': 798.3668, 'train_samples_per_second': 143.986, 'train_steps_per_second': 9.001, 'total_flos': 1.5227928908752896e+16, 'train_loss': 1.0682368545256185, 'epoch': 2.0})

In [41]:
def combine(sample):
    return {"text": f"<prompt>{sample["prompt"]}\n\n<answer1>{sample["response_a"]}\n\n<answer2>{sample["response_b"]}"}

test_ds = dataset["test"].map(combine)

In [61]:
import torch
from collections import defaultdict

outs = defaultdict(list)

model = model.cpu()
for sample in test_ds:
    tokenized = tokenizer(sample["text"], return_tensors="pt", truncation=True)
    # tokenized = tokenized.to("cuda:0")
    with torch.no_grad():
        logits = model(**tokenized).logits
    probs = logits.softmax(dim=-1)[0].tolist()
    outs["id"].append(sample["id"])
    outs["winner_model_a"].append(probs[0])
    outs["winner_model_b"].append(probs[2])
    outs["tie"].append(probs[1])

defaultdict(list,
            {'id': [136060, 211333, 1233961],
             'winner_model_a': [0.23093360662460327,
              0.3105784058570862,
              0.35215452313423157],
             'winner_model_b': [0.22283044457435608,
              0.43917638063430786,
              0.37008363008499146],
             'tie': [0.546235978603363,
              0.25024521350860596,
              0.2777618169784546]})

In [63]:
import pandas as pd

outs = pd.DataFrame(outs)
outs.to_csv("submission.csv")

In [64]:
type(tokenizer)

transformers.models.distilbert.tokenization_distilbert_fast.DistilBertTokenizerFast